In [ ]:
import pandas as pd

# Load the dataset directly (replace path as needed)
file_path = 'NCAA_Seed_Training_Set2.0.csv'
df = pd.read_csv(file_path)

print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
print(df.head())

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1353 entries, 0 to 1352
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   RecordID              1353 non-null   object 
 1   Season                1353 non-null   object 
 2   Team                  1353 non-null   object 
 3   Conference            1353 non-null   object 
 4   Overall Seed          249 non-null    float64
 5   Bid Type              249 non-null    object 
 6   NET Rank              1348 non-null   float64
 7   PrevNET               1348 non-null   float64
 8   AvgOppNETRank         1348 non-null   float64
 9   AvgOppNET             1348 non-null   float64
 10  WL                    1353 non-null   object 
 11  Conf.Record           1353 non-null   object 
 12  Non-ConferenceRecord  1353 non-null   object 
 13  RoadWL                1353 non-null   object 
 14  NETSOS                1348 non-null   float64
 15  NETNonConfSOS        

In [ ]:
duplicate_rows = df[df.duplicated()]
num_duplicates = duplicate_rows.shape[0]

if num_duplicates > 0:
    print(f"Found {num_duplicates} duplicate rows in the DataFrame.")

else:
    print("No duplicate rows found in the DataFrame.")

No duplicate rows found in the DataFrame.


In [ ]:
import pandas as pd
import re


# Columns that should have win-loss or quadrant records
wl_columns = [
    'WL','Conf.Record','Non-ConferenceRecord','RoadWL',
    'Quadrant1','Quadrant2','Quadrant3','Quadrant4'
]

# Map month abbreviations to numbers
month_map = {
    'Jan':'1', 'Feb':'2', 'Mar':'3', 'Apr':'4', 'May':'5', 'Jun':'6',
    'Jul':'7', 'Aug':'8', 'Sep':'9', 'Oct':'10', 'Nov':'11', 'Dec':'12'
}

def recover_win_loss(value):
    if pd.isna(value):
        return value

    value = value.strip()

    # Case 1: already a proper win-loss (e.g., 15-12, 0-0)
    if re.match(r'^\d+-\d+$', value):
        return value

    # Case 2: Excel converted like '8-May', '10-Dec'
    match = re.match(r'^(\d+)-([A-Za-z]{3})$', value)
    if match:
        wins = match.group(1)
        month_abbr = match.group(2)
        if month_abbr in month_map:
            losses = month_map[month_abbr]
            return f"{wins}-{losses}"

    # Case 3: Excel converted like 'Jun-00'
    match2 = re.match(r'^([A-Za-z]{3})-(\d+)$', value)
    if match2:
        month_abbr = match2.group(1)
        year_part = match2.group(2)
        # Most likely the original win was 0, loss = year or 0
        losses = int(year_part)
        return f"0-{losses}"

    # If none matched, leave as is
    return value

# Apply recovery function to all relevant columns
for col in wl_columns:
    if col in df.columns:
        df[col] = df[col].apply(recover_win_loss)

# Check results
print(df[wl_columns].head())

      WL Conf.Record Non-ConferenceRecord RoadWL Quadrant1 Quadrant2  \
0    8-9         5-7                  3-2    3-4       0-0       0-3   
1  15-12        12-9                  0-0    8-3      10-5       2-4   
2   19-7        14-5                  5-2   10-2       0-0       0-1   
3   16-5        13-5                  0-3    9-2       0-1       0-2   
4   14-1        14-1                  0-0    0-0       0-0       0-0   

  Quadrant3 Quadrant4  
0       2-1       3-8  
1       0-0       0-0  
2       7-2      11-5  
3       6-2       7-3  
4       1-6       0-0  


In [ ]:
df.head()

,RecordID,Season,Team,Conference,Overall Seed,Bid Type,NET Rank,PrevNET,AvgOppNETRank,AvgOppNET,WL,Conf.Record,Non-ConferenceRecord,RoadWL,NETSOS,NETNonConfSOS,Quadrant1,Quadrant2,Quadrant3,Quadrant4
0,2020-21-UCDavis,2020-21,UC Davis,Big West,NaN,NaN,223.0,224.0,240.0,211.0,8-9,5-7,3-2,3-4,264.0,287.0,0-0,0-3,2-1,3-8
1,2020-21-MichiganSt.,2020-21,Michigan St.,Big Ten,43.0,AL,70.0,70.0,20.0,75.0,15-12,12-9,0-0,8-3,9.0,238.0,10-5,2-4,0-0,0-0
2,2020-21-ULM,2020-21,ULM,Sun Belt,NaN,NaN,292.0,292.0,244.0,214.0,19-7,14-5,5-2,10-2,313.0,263.0,0-0,0-1,7-2,11-5
3,2020-21-CentralConn.St.,2020-21,Central Conn. St.,NEC,NaN,NaN,301.0,301.0,200.0,195.0,16-5,13-5,0-3,9-2,242.0,54.0,0-1,0-2,6-2,7-3
4,2020-21-Colgate,2020-21,Colgate,Patriot,57.0,AQ,9.0,9.0,154.0,169.0,14-1,14-1,0-0,0-0,257.0,NaN,0-0,0-0,1-6,0-0


In [ ]:
def parse_win_loss_string(wl_string):
    if pd.isna(wl_string):
        return None, None
    parts = str(wl_string).split('-')
    if len(parts) == 2:
        try:
            wins = int(parts[0])
            losses = int(parts[1])
            return wins, losses
        except ValueError:
            return None, None
    return None, None

# Iterate through the wl_columns and apply the transformation
for col in wl_columns:
    if col in df.columns:
        # Create new Wins and Losses columns
        df[[f'{col}_Wins', f'{col}_Losses']] = df[col].apply(lambda x: pd.Series(parse_win_loss_string(x)))

        # Calculate WinPercentage
        total_games = df[f'{col}_Wins'] + df[f'{col}_Losses']
        df[f'{col}_WinPercentage'] = df[f'{col}_Wins'] / total_games
        # Handle cases where total_games is 0 to avoid division by zero and set WinPercentage to 0
        df.loc[total_games == 0, f'{col}_WinPercentage'] = 0

        # Drop the original column
        df = df.drop(columns=[col])

print(df.head())

                  RecordID   Season               Team Conference  \
0          2020-21-UCDavis  2020-21           UC Davis   Big West   
1      2020-21-MichiganSt.  2020-21       Michigan St.    Big Ten   
2              2020-21-ULM  2020-21                ULM   Sun Belt   
3  2020-21-CentralConn.St.  2020-21  Central Conn. St.        NEC   
4          2020-21-Colgate  2020-21            Colgate    Patriot   

   Overall Seed Bid Type  NET Rank  PrevNET  AvgOppNETRank  AvgOppNET  ...  \
0           NaN      NaN     223.0    224.0          240.0      211.0  ...   
1          43.0       AL      70.0     70.0           20.0       75.0  ...   
2           NaN      NaN     292.0    292.0          244.0      214.0  ...   
3           NaN      NaN     301.0    301.0          200.0      195.0  ...   
4          57.0       AQ       9.0      9.0          154.0      169.0  ...   

   Quadrant1_WinPercentage  Quadrant2_Wins  Quadrant2_Losses  \
0                 0.000000               0          

In [ ]:
print(df['Overall Seed'].describe())
print(df['Overall Seed'].unique())
print(f"Number of missing values in 'Overall Seed': {df['Overall Seed'].isnull().sum()}")

count    249.000000
mean      33.377510
std       19.394931
min        1.000000
25%       17.000000
50%       32.000000
75%       49.000000
max       68.000000
Name: Overall Seed, dtype: float64
[nan 43. 57. 16. 58. 34. 26. 10. 40. 23. 31. 56.  6. 46.  4. 64. 47. 17.
 67.  8. 61. 18. 25. 20. 19. 32. 37. 60.  7. 11. 24. 38. 68. 39. 12.  3.
 30. 45. 66. 13. 36. 29. 52. 42. 27.  5.  1. 62. 48. 33. 28. 55. 54. 50.
 44. 63. 22. 15.  9. 21. 53. 35. 59. 14. 41.  2. 49. 51. 65.]
Number of missing values in 'Overall Seed': 1104


In [ ]:
df.dropna(subset=['Overall Seed'], inplace=True)
df['Overall Seed'] = df['Overall Seed'].astype(int)

print('DataFrame Info after processing "Overall Seed":')
df.info()
print('\nDataFrame Head after processing "Overall Seed":')
print(df.head())

DataFrame Info after processing "Overall Seed":
<class 'pandas.core.frame.DataFrame'>
Index: 249 entries, 1 to 1351
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   RecordID                            249 non-null    object 
 1   Season                              249 non-null    object 
 2   Team                                249 non-null    object 
 3   Conference                          249 non-null    object 
 4   Overall Seed                        249 non-null    int64  
 5   Bid Type                            249 non-null    object 
 6   NET Rank                            249 non-null    float64
 7   PrevNET                             249 non-null    float64
 8   AvgOppNETRank                       249 non-null    float64
 9   AvgOppNET                           249 non-null    float64
 10  NETSOS                              249 non-null    float64
 11  N

In [ ]:
print(df.isnull().sum())

RecordID                              0
Season                                0
Team                                  0
Conference                            0
Overall Seed                          0
Bid Type                              0
NET Rank                              0
PrevNET                               0
AvgOppNETRank                         0
AvgOppNET                             0
NETSOS                                0
NETNonConfSOS                         1
WL_Wins                               0
WL_Losses                             0
WL_WinPercentage                      0
Conf.Record_Wins                      0
Conf.Record_Losses                    0
Conf.Record_WinPercentage             0
Non-ConferenceRecord_Wins             0
Non-ConferenceRecord_Losses           0
Non-ConferenceRecord_WinPercentage    0
RoadWL_Wins                           0
RoadWL_Losses                         0
RoadWL_WinPercentage                  0
Quadrant1_Wins                        0


In [ ]:
df['NETNonConfSOS'] = df['NETNonConfSOS'].fillna(df['NETNonConfSOS'].mean())
print("Missing values in 'NETNonConfSOS' imputed with its mean.")

Missing values in 'NETNonConfSOS' imputed with its mean.


In [ ]:
print(df.isnull().sum())

RecordID                              0
Season                                0
Team                                  0
Conference                            0
Overall Seed                          0
Bid Type                              0
NET Rank                              0
PrevNET                               0
AvgOppNETRank                         0
AvgOppNET                             0
NETSOS                                0
NETNonConfSOS                         0
WL_Wins                               0
WL_Losses                             0
WL_WinPercentage                      0
Conf.Record_Wins                      0
Conf.Record_Losses                    0
Conf.Record_WinPercentage             0
Non-ConferenceRecord_Wins             0
Non-ConferenceRecord_Losses           0
Non-ConferenceRecord_WinPercentage    0
RoadWL_Wins                           0
RoadWL_Losses                         0
RoadWL_WinPercentage                  0
Quadrant1_Wins                        0


In [ ]:
df_encoded = pd.get_dummies(df, columns=['Conference', 'Bid Type'], drop_first=True)
df = df_encoded

print("DataFrame head after one-hot encoding:")
print(df.head())

print("\nDataFrame info after one-hot encoding:")
df.info()

DataFrame head after one-hot encoding:
                RecordID   Season           Team  Overall Seed  NET Rank  \
1    2020-21-MichiganSt.  2020-21   Michigan St.            43      70.0   
4        2020-21-Colgate  2020-21        Colgate            57       9.0   
6       2020-21-Virginia  2020-21       Virginia            16      12.0   
7   2020-21-EasternWash.  2020-21  Eastern Wash.            58     114.0   
23   2020-21-GeorgiaTech  2020-21   Georgia Tech            34      34.0   

    PrevNET  AvgOppNETRank  AvgOppNET  NETSOS  NETNonConfSOS  ...  \
1      70.0           20.0       75.0     9.0     238.000000  ...   
4       9.0          154.0      169.0   257.0     137.060484  ...   
6      12.0           57.0      100.0    49.0     147.000000  ...   
7     113.0          252.0      216.0   217.0      12.000000  ...   
23     33.0           49.0       95.0    48.0     236.000000  ...   

    Conference_SEC  Conference_SWAC  Conference_SoCon  Conference_Southland  \
1         

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Separate features (X) and target variable (y)
target = 'Overall Seed'
identifier_cols = ['RecordID', 'Season', 'Team']

X = df.drop(columns=[target] + identifier_cols)
y = df[target]

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shapes of the datasets after splitting:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Shapes of the datasets after splitting:
X_train shape: (199, 62)
X_test shape: (50, 62)
y_train shape: (199,)
y_test shape: (50,)


## Model Training

**Note:** `Overall Seed` (1–68) is an **ordinal/continuous** target — not independent categories.
Using `XGBRegressor` is the correct choice here. `XGBClassifier` requires contiguous 0-indexed classes,
which fails during cross-validation when some seed values are missing from a fold.


In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Separate features and target
target = 'Overall Seed'
identifier_cols = ['RecordID', 'Season', 'Team']

X = df.drop(columns=[target] + identifier_cols)
y = df[target]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

# Train XGBRegressor
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)

# Evaluate
y_pred = xgb_model.predict(X_test)
y_pred_rounded = np.round(y_pred).astype(int).clip(1, 68)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.4f}')
print(f'\nSample predictions vs actuals:')
comparison = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred_rounded[:len(y_test)]})
print(comparison.head(10))

## Hyperparameter Tuning

Using `GridSearchCV` with `XGBRegressor` and `KFold`. 
Note: `GridSearchCV` does **not** accept a `random_state` parameter — that was a bug in the original code.


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from xgboost import XGBRegressor

param_grid = {
    'n_estimators':  [50, 100, 200],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample':     [0.7, 0.9, 1.0]
}

xgb_reg = XGBRegressor(random_state=42)

# KFold avoids the stratification issue that broke the original Classifier approach
kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_reg,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=kf,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print(f'Best CV MAE: {-grid_search.best_score_:.2f}')

# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(f'\nTest MAE  (tuned): {mean_absolute_error(y_test, y_pred_best):.2f}')
print(f'Test RMSE (tuned): {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f}')
print(f'Test R²   (tuned): {r2_score(y_test, y_pred_best):.4f}')

## Test Set Prediction & Submission

### Key insight
The test set has **451 rows** but only **91 are tournament teams** (have a `Bid Type`).
The other 360 teams did not make the tournament and have **no seed** (observed = 0 / NaN).
Predicting a non-zero seed for non-tournament teams causes massive RMSE inflation.

**Fix:** predict 0 for non-tournament teams, run the model only on tournament teams.


In [ ]:
import pandas as pd
import numpy as np
import re

test_df = pd.read_csv('NCAA_Seed_Test_Set2_0.csv')
print(f'Test set: {len(test_df)} total rows')
print(f'  Tournament teams (Bid Type set): {test_df["Bid Type"].notna().sum()}')
print(f'  Non-tournament teams (no Bid Type): {test_df["Bid Type"].isna().sum()}')


In [ ]:
# Same WL recovery pipeline as training
month_map = {
    'Jan':'1','Feb':'2','Mar':'3','Apr':'4','May':'5','Jun':'6',
    'Jul':'7','Aug':'8','Sep':'9','Oct':'10','Nov':'11','Dec':'12'
}

def recover_win_loss(value):
    if pd.isna(value): return value
    value = str(value).strip()
    if re.match(r'^\d+-\d+$', value): return value
    m = re.match(r'^(\d+)-([A-Za-z]{3})$', value)
    if m: return f"{m.group(1)}-{month_map.get(m.group(2), m.group(2))}"
    m2 = re.match(r'^([A-Za-z]{3})-(\d+)$', value)
    if m2: return f"0-{int(m2.group(2))}"
    return value

def parse_wl(wl):
    if pd.isna(wl): return None, None
    parts = str(wl).split('-')
    try: return int(parts[0]), int(parts[1])
    except: return None, None

wl_columns = ['WL','Conf.Record','Non-ConferenceRecord','RoadWL',
               'Quadrant1','Quadrant2','Quadrant3','Quadrant4']

for col in wl_columns:
    if col in test_df.columns:
        test_df[col] = test_df[col].apply(recover_win_loss)

for col in wl_columns:
    if col in test_df.columns:
        test_df[[f'{col}_Wins', f'{col}_Losses']] = test_df[col].apply(
            lambda x: pd.Series(parse_wl(x)))
        total = test_df[f'{col}_Wins'] + test_df[f'{col}_Losses']
        test_df[f'{col}_WinPercentage'] = test_df[f'{col}_Wins'] / total
        test_df.loc[total == 0, f'{col}_WinPercentage'] = 0
        test_df = test_df.drop(columns=[col])

# Use training mean for imputation (not test mean — avoids data leakage)
train_nonconf_mean = df['NETNonConfSOS'].mean()
test_df['NETNonConfSOS'] = test_df['NETNonConfSOS'].fillna(train_nonconf_mean)

print('Preprocessing complete.')
print(f'Remaining nulls: {test_df.isnull().sum()[test_df.isnull().sum()>0].to_dict()}')


In [ ]:
# ── One-hot encode ──
test_encoded = pd.get_dummies(test_df, columns=['Conference', 'Bid Type'], drop_first=True)

# ── Split: tournament teams only (these have seeds to predict) ──
# 'Bid Type_AQ' column will exist only if AQ teams are present
# We identify tournament teams by checking original Bid Type
tourney_mask = test_df['Bid Type'].notna() if 'Bid Type' in test_df.columns else \
               test_encoded.get('Bid Type_AQ', pd.Series(False, index=test_encoded.index)) | \
               test_encoded.get('Bid Type_AL', pd.Series(False, index=test_encoded.index))

# Re-derive from original (safer)
original_test = pd.read_csv('NCAA_Seed_Test_Set2_0.csv')
tourney_mask = original_test['Bid Type'].notna().values

record_ids_all = test_encoded['RecordID'].values
identifier_cols = ['RecordID', 'Season', 'Team']
X_test_all = test_encoded.drop(columns=[c for c in identifier_cols if c in test_encoded.columns])

# Align columns to training feature set
for col in X_train.columns:
    if col not in X_test_all.columns:
        X_test_all[col] = 0
X_test_all = X_test_all[X_train.columns]

# Only predict on tournament teams
X_tourney = X_test_all[tourney_mask]
record_ids_tourney = record_ids_all[tourney_mask]
record_ids_non = record_ids_all[~tourney_mask]

print(f'Tournament teams to predict: {len(X_tourney)}')
print(f'Non-tournament teams (predict 0): {(~tourney_mask).sum()}')

# Predict seeds for tournament teams
y_pred_tourney = best_model.predict(X_tourney)
y_pred_tourney_rounded = np.round(y_pred_tourney).astype(int).clip(1, 68)

print(f'\nTournament seed predictions:')
print(f'  Min: {y_pred_tourney_rounded.min()}')
print(f'  Max: {y_pred_tourney_rounded.max()}')
print(f'  Mean: {y_pred_tourney_rounded.mean():.1f}')


In [ ]:
# ── Build submission ──
submission = pd.read_csv('submission_template2_0.csv')

# Map tournament team predictions
pred_map = dict(zip(record_ids_tourney, y_pred_tourney_rounded))

# Non-tournament teams get 0 (they have no seed)
non_tourney_map = dict(zip(record_ids_non, [0] * len(record_ids_non)))
pred_map.update(non_tourney_map)

submission['Overall Seed'] = submission['RecordID'].map(pred_map)

print('Seed distribution in submission:')
print(f'  Rows with seed > 0 (tournament): {(submission["Overall Seed"] > 0).sum()}')
print(f'  Rows with seed = 0 (non-tourney): {(submission["Overall Seed"] == 0).sum()}')
print(f'  Null rows: {submission["Overall Seed"].isna().sum()}')

submission.to_csv('submission.csv', index=False)
print(f'\nSaved submission.csv ({len(submission)} rows)')
print(submission[submission['Overall Seed'] > 0].head(15))


In [ ]:
# ── Sanity check: predicted seeds vs NET Rank ──
# Higher NET rank = worse team, should get higher seed number
check = original_test[tourney_mask][['Team','Season','Bid Type','NET Rank']].copy()
check['Predicted Seed'] = y_pred_tourney_rounded
check = check.sort_values('Predicted Seed')
print('Top 20 predicted seeds (should correlate with low NET Rank):')
print(check.head(20).to_string(index=False))
print('\nBottom 10 predicted seeds:')
print(check.tail(10).to_string(index=False))
